# CSC 485B Assignment 2: Hierarchical Parallelism (Group 2)

Starter notebook with our code added. `data_types.h` and `data_generator.h` are unchanged; we added code to `cuda_common.h`, `dense_graph.h`, `sparse_graph.h` and `main.cu`, and added `cpu_reference.h` for testing.

| Tier | What we did |
|---|---|
| 50 / 55 | `build()` for DenseGraph (memset + grid-stride scatter) and SparseGraph (atomic degree count, prefix sum over warp/block/grid, atomic scatter, sort per node), both with many blocks |
| 60 / 70 | Naive two-hop: one thread per output cell (n² threads, 16×16 blocks) |
| 75 | Tiled two-hop: 16×16 tiles in shared memory with `__syncthreads()` |
| 80 | Warp-level two-hop: rows packed into bits with `__ballot_sync`, row words shared with `__shfl_sync`, AND/OR instead of multiply-add |
| 85 | Explanation of why the warp-level version uses the hardware better, with Nsight Compute numbers (last section and video) |

We did not do the two-hop query on the SparseGraph (A+ / 100 tiers).

**How to run:** T4 runtime, run all cells. The run cell prints the example from the assignment, a correctness check against a CPU version on 11 graphs, and a benchmark at n = 4096.

In [ ]:
# Load the extension that allows us to compile CUDA code in python notebooks
# Documentation is here: https://nvcc4jupyter.readthedocs.io/en/latest/
!pip install git+https://github.com/andreinechaev/nvcc4jupyter.git
%load_ext nvcc4jupyter

In [ ]:
%%cuda_group_save -g "source" -n "data_types.h"
/**
 * A collection of commonly used data types throughout this project.
 */
#pragma once

#include <iostream> // for std::ostream
#include <vector>

namespace csc485b{
namespace a2{

using node_t = int;
using edge_t = int2;

using edge_list_t = std::vector< edge_t >;
using node_list_t = std::vector< node_t >;

} // namespace a2
} // namespace csc485b

In [ ]:
%%cuda_group_save -g "source" -n "cuda_common.h"
/**
 * Standard macros that can be useful for error checking.
 * https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__ERROR.html
 */
#pragma once

#include <cuda.h>

#define CUDA_CALL(exp)                                       \
    do {                                                     \
        cudaError res = (exp);                               \
        if(res != cudaSuccess) {                             \
            printf("Error at %s:%d\n %s\n",                  \
                __FILE__,__LINE__, cudaGetErrorString(res)); \
           exit(EXIT_FAILURE);                               \
        }                                                    \
    } while(0)

#define CHECK_ERROR(msg)                                             \
    do {                                                             \
        cudaError_t err = cudaGetLastError();                        \
        if(cudaSuccess != err) {                                     \
            printf("Error (%s) at %s:%d\n %s\n",                     \
                (msg), __FILE__, __LINE__, cudaGetErrorString(err)); \
            exit(EXIT_FAILURE);                                      \
        }                                                            \
    } while (0)

#include <algorithm> // std::min, std::max
#include <cstddef>   // std::size_t

/** Threads per block used by the grid-stride kernels. */
constexpr unsigned int DEFAULT_BLOCK_SIZE = 256;

/**
 * Number of blocks to launch for a grid-stride loop over `work` items:
 * enough to cover all of the work (at least one block), capped so that
 * very large inputs are handled by looping instead.
 */
inline unsigned int num_blocks_for( std::size_t work, unsigned int block_size )
{
    std::size_t const blocks = ( work + block_size - 1 ) / block_size;
    return static_cast< unsigned int >( std::min< std::size_t >( std::max< std::size_t >( blocks, 1 ), 65535 ) );
}

In [ ]:
%%cuda_group_save -g "source" -n "data_generator.h"
/**
 * Functions for generating random input data with a fixed seed
 */
#pragma once

#include <cassert>  // for assert()
#include <cstddef>  // std::size_t type
#include <random>   // for std::mt19937, std::uniform_int_distribution
#include <vector>

#include "data_types.h"

namespace csc485b {
namespace a2 {

/**
 * Generates and returns a vector of random edges
 * for a graph `G=(V,E)` with `n=|V|=n` and expected `m=|E|`.
 * Referred to as an Erdős-Rényi graph.
 *
 * @see https://networkx.org/documentation/stable/reference/generated/networkx.generators.random_graphs.fast_gnp_random_graph.html#networkx.generators.random_graphs.fast_gnp_random_graph
 */
edge_list_t generate_graph( std::size_t n, std::size_t m )
{
    assert( "At most n(n-1) edges in a simple graph" && m < n * ( n - 1 ) );

    int const probability = ( 100 * m ) / ( n * ( n - 1 ) );

    // for details of random number generation, see:
    // https://en.cppreference.com/w/cpp/numeric/random/uniform_int_distribution
    std::size_t random_seed = 20241008;  // use magic seed
    std::mt19937 rng( random_seed );     // use mersenne twister generator
    std::uniform_int_distribution<> distrib(0, 100);

    edge_list_t random_edges;
    random_edges.reserve( 2 * m );

    for( node_t u = 0; u < n; ++u )
    {
        for( node_t v = u + 1; v < n; ++v )
        {
            auto const dice_roll = distrib( rng );
            if( dice_roll <= probability )
            {
                random_edges.push_back( make_int2( u, v ) );
                random_edges.push_back( make_int2( v, u ) );
            }
        }
    }

    random_edges.resize( random_edges.size() );


    return random_edges;
}

} // namespace a2
} // namespace csc485b

In [ ]:
%%cuda_group_save -g "source" -n "dense_graph.h"
/**
 * The file in which you will implement your DenseGraph GPU solutions!
 */

#include <cstddef>  // std::size_t type

#include "cuda_common.h"
#include "data_types.h"

namespace csc485b {
namespace a2      {

/**
 * A DenseGraph is optimised for a graph in which the number of edges
 * is close to n(n-1). It is represented using an adjacency matrix.
 */
struct DenseGraph
{
  std::size_t n; /**< Number of nodes in the graph. */
  node_t * adjacencyMatrix; /** Pointer to an n x n adj. matrix */

  /** Returns number of cells in the adjacency matrix. */
  __device__ __host__ __forceinline__
  std::size_t matrix_size() const { return n * n; }
};


namespace gpu {


/**
 * Constructs a DenseGraph from an input edge list of m edges.
 *
 * @pre The pointers in DenseGraph g have already been allocated.
 * @pre The adjacency matrix has been zeroed (see build() below).
 */
__global__
void build_graph( DenseGraph g, edge_t const * edge_list, std::size_t m )
{
    // grid-stride loop, one thread per edge. Every thread only ever writes 1s,
    // so concurrent writes to the same cell are harmless.
    std::size_t const stride = static_cast< std::size_t >( gridDim.x ) * blockDim.x;
    for( std::size_t e = static_cast< std::size_t >( blockIdx.x ) * blockDim.x + threadIdx.x; e < m; e += stride )
    {
        edge_t const edge = edge_list[ e ];
        g.adjacencyMatrix[ edge.x * g.n + edge.y ] = 1;
    }
}

/**
  * Repopulates the adjacency matrix as a new graph that represents
  * the two-hop neighbourhood of input graph g
  *
  * Naive version (C / B- tier): one thread per output cell (i,j), each reading
  * row i and column j straight from global memory. The result goes to a separate
  * matrix because other threads are still reading g.
  */
__global__
void two_hop_reachability( DenseGraph g, node_t * result )
{
    std::size_t const n = g.n;
    std::size_t const i = static_cast< std::size_t >( blockIdx.y ) * blockDim.y + threadIdx.y;
    std::size_t const j = static_cast< std::size_t >( blockIdx.x ) * blockDim.x + threadIdx.x;
    if( i >= n || j >= n ) return;

    // square adjacencyMatrix
    node_t paths = 0;
    for( std::size_t k = 0; k < n; ++k )
    {
        paths += g.adjacencyMatrix[ i * n + k ] * g.adjacencyMatrix[ k * n + j ];
    }

    // then remove the diagonal and clamp values back to [0,1]
    result[ i * n + j ] = ( i != j && paths > 0 );
}

/** Side length of the square shared-memory tiles of the tiled version. */
constexpr int TILE = 16;

/**
 * Tiled version (B tier): a TILE x TILE block stages a tile of rows and a tile
 * of columns in shared memory, so every value loaded from global memory is
 * reused TILE times. __syncthreads() separates the load and compute phases.
 */
__global__
void two_hop_reachability_tiled( DenseGraph g, node_t * result )
{
    __shared__ node_t row_tile[ TILE ][ TILE ];
    __shared__ node_t col_tile[ TILE ][ TILE ];

    std::size_t const n = g.n;
    int const tx = threadIdx.x;
    int const ty = threadIdx.y;
    std::size_t const i = static_cast< std::size_t >( blockIdx.y ) * TILE + ty;
    std::size_t const j = static_cast< std::size_t >( blockIdx.x ) * TILE + tx;

    node_t paths = 0;
    for( std::size_t k0 = 0; k0 < n; k0 += TILE )
    {
        // cells outside the matrix load as 0 so they do not contribute
        row_tile[ ty ][ tx ] = ( i < n && k0 + tx < n ) ? g.adjacencyMatrix[ i * n + k0 + tx ] : 0;
        col_tile[ ty ][ tx ] = ( k0 + ty < n && j < n ) ? g.adjacencyMatrix[ ( k0 + ty ) * n + j ] : 0;
        __syncthreads();

        #pragma unroll
        for( int k = 0; k < TILE; ++k )
        {
            paths += row_tile[ ty ][ k ] * col_tile[ k ][ tx ];
        }
        __syncthreads(); // nobody overwrites the tiles while others still read them
    }

    if( i < n && j < n )
    {
        result[ i * n + j ] = ( i != j && paths > 0 );
    }
}

/** Number of adjacency-matrix cells packed into one bitmask word. */
constexpr int WORD_BITS = 32;

/**
 * Packs every row of the adjacency matrix into ceil(n/32) bitmask words.
 * One warp per word: lane l tests cell (row, 32w + l) and __ballot_sync
 * gathers the 32 predicates into a single register.
 */
__global__
void pack_rows( DenseGraph g, unsigned int * bits, std::size_t words_per_row )
{
    unsigned int const lane = threadIdx.x % warpSize;
    std::size_t const warp      = ( static_cast< std::size_t >( blockIdx.x ) * blockDim.x + threadIdx.x ) / warpSize;
    std::size_t const num_warps = ( static_cast< std::size_t >( gridDim.x ) * blockDim.x ) / warpSize;

    // the loop condition only depends on the warp index, so whole warps
    // reach the ballot together, as the full mask requires
    for( std::size_t word = warp; word < g.n * words_per_row; word += num_warps )
    {
        std::size_t const row = word / words_per_row;
        std::size_t const col = ( word % words_per_row ) * WORD_BITS + lane;
        bool const edge = col < g.n && g.adjacencyMatrix[ row * g.n + col ] != 0;
        unsigned int const packed = __ballot_sync( 0xffffffff, edge );
        if( lane == 0 ) bits[ word ] = packed;
    }
}

/** Output rows computed by each warp of the warp-level version. */
constexpr int ROWS_PER_WARP = 4;
/** Warps per block of the warp-level version; a block covers a 32 x 32 output tile. */
constexpr int WARPS_PER_BLOCK = WORD_BITS / ROWS_PER_WARP;

/**
 * Warp-level version (A- tier): boolean matrix multiplication on bit-packed rows.
 * The graph is undirected, so column j equals row j, and (A^2)[i][j] > 0 iff
 * rows i and j share a set bit: one AND covers 32 values of k.
 *
 * A block computes a 32 x 32 output tile. For each chunk of 32 words, the block
 * stages the 32 column-rows in shared memory (padded against bank conflicts),
 * while each warp keeps its own rows in registers, one word per lane, and
 * broadcasts them with __shfl_sync instead of going through shared memory.
 * Lane l owns output column j0 + l, so the final writes are coalesced.
 * The result can go straight into g because only the packed copy is read.
 */
__global__
void two_hop_reachability_warp( DenseGraph g, unsigned int const * bits, std::size_t words_per_row )
{
    __shared__ unsigned int col_words[ WORD_BITS ][ WORD_BITS + 1 ];

    std::size_t const n = g.n;
    unsigned int const lane = threadIdx.x % warpSize;
    unsigned int const warp = threadIdx.x / warpSize;
    std::size_t const i0 = static_cast< std::size_t >( blockIdx.y ) * WORD_BITS + warp * ROWS_PER_WARP;
    std::size_t const j0 = static_cast< std::size_t >( blockIdx.x ) * WORD_BITS;

    unsigned int reachable[ ROWS_PER_WARP ] = { 0 };
    for( std::size_t w0 = 0; w0 < words_per_row; w0 += WORD_BITS )
    {
        std::size_t const w = w0 + lane;

        // stage word w of rows j0..j0+31 (consecutive lanes read consecutive words)
        for( unsigned int r = warp; r < WORD_BITS; r += WARPS_PER_BLOCK )
        {
            col_words[ r ][ lane ] = ( j0 + r < n && w < words_per_row ) ? bits[ ( j0 + r ) * words_per_row + w ] : 0;
        }

        // this warp's own rows stay in registers: lane holds word w of each row
        unsigned int row_word[ ROWS_PER_WARP ];
        #pragma unroll
        for( int r = 0; r < ROWS_PER_WARP; ++r )
        {
            row_word[ r ] = ( i0 + r < n && w < words_per_row ) ? bits[ ( i0 + r ) * words_per_row + w ] : 0;
        }
        __syncthreads();

        #pragma unroll 8
        for( int t = 0; t < WORD_BITS; ++t )
        {
            unsigned int const col = col_words[ lane ][ t ]; // bank (lane + t) % 32: conflict-free
            #pragma unroll
            for( int r = 0; r < ROWS_PER_WARP; ++r )
            {
                reachable[ r ] |= __shfl_sync( 0xffffffff, row_word[ r ], t ) & col;
            }
        }
        __syncthreads();
    }

    // remove the diagonal and clamp values back to [0,1]
    std::size_t const j = j0 + lane;
    #pragma unroll
    for( int r = 0; r < ROWS_PER_WARP; ++r )
    {
        std::size_t const i = i0 + r;
        if( i < n && j < n )
        {
            g.adjacencyMatrix[ i * n + j ] = ( i != j && reachable[ r ] != 0 );
        }
    }
}

/** The DenseGraph two-hop implementations, in the order of the rubric tiers. */
enum class DenseAlgorithm { naive, tiled, warp };

/**
 * Host-side driver for build_graph(): zeroes the matrix, then scatters
 * the edges using as many blocks as needed to cover them.
 */
inline void build( DenseGraph & g, edge_t const * edge_list, std::size_t m )
{
    CUDA_CALL( cudaMemset( g.adjacencyMatrix, 0, sizeof( node_t ) * g.matrix_size() ) );
    build_graph<<< num_blocks_for( m, DEFAULT_BLOCK_SIZE ), DEFAULT_BLOCK_SIZE >>>( g, edge_list, m );
    CHECK_ERROR( "build_graph(DenseGraph)" );
}

/** Host-side driver for the two-hop query: replaces g's matrix with its two-hop neighbourhood. */
inline void two_hop( DenseGraph & g, DenseAlgorithm algorithm = DenseAlgorithm::warp )
{
    std::size_t const n = g.n;

    if( algorithm == DenseAlgorithm::warp )
    {
        std::size_t const words_per_row = ( n + WORD_BITS - 1 ) / WORD_BITS;
        unsigned int * d_bits;
        CUDA_CALL( cudaMalloc( (void**)&d_bits, sizeof( unsigned int ) * n * words_per_row ) );

        pack_rows<<< num_blocks_for( n * words_per_row * WORD_BITS, DEFAULT_BLOCK_SIZE ), DEFAULT_BLOCK_SIZE >>>( g, d_bits, words_per_row );
        CHECK_ERROR( "pack_rows" );

        unsigned int const tiles = static_cast< unsigned int >( words_per_row );
        two_hop_reachability_warp<<< dim3( tiles, tiles ), WARPS_PER_BLOCK * WORD_BITS >>>( g, d_bits, words_per_row );
        CHECK_ERROR( "two_hop_reachability_warp" );

        CUDA_CALL( cudaFree( d_bits ) );
        return;
    }

    node_t * d_result;
    CUDA_CALL( cudaMalloc( (void**)&d_result, sizeof( node_t ) * g.matrix_size() ) );

    unsigned int const tiles = static_cast< unsigned int >( ( n + TILE - 1 ) / TILE );
    if( algorithm == DenseAlgorithm::naive )
    {
        two_hop_reachability<<< dim3( tiles, tiles ), dim3( TILE, TILE ) >>>( g, d_result );
    }
    else
    {
        two_hop_reachability_tiled<<< dim3( tiles, tiles ), dim3( TILE, TILE ) >>>( g, d_result );
    }
    CHECK_ERROR( "two_hop_reachability(DenseGraph)" );

    CUDA_CALL( cudaMemcpy( g.adjacencyMatrix, d_result, sizeof( node_t ) * g.matrix_size(), cudaMemcpyDeviceToDevice ) );
    CUDA_CALL( cudaFree( d_result ) );
}

} // namespace gpu
} // namespace a2
} // namespace csc485b

In [ ]:
%%cuda_group_save -g "source" -n "sparse_graph.h"
/**
 * The file in which you will implement your SparseGraph GPU solutions!
 */

#include <cstddef>  // std::size_t type

#include "cuda_common.h"
#include "data_types.h"

namespace csc485b {
namespace a2      {

/**
 * A SparseGraph is optimised for a graph in which the number of edges
 * is close to cn, for a small constanct c. It is represented in CSR format.
 */
struct SparseGraph
{
  std::size_t n; /**< Number of nodes in the graph. */
  std::size_t m; /**< Number of edges in the graph. */
  node_t * neighbours_start_at; /** Pointer to an n=|V| offset array */
  node_t * neighbours; /** Pointer to an m=|E| array of edge destinations */
};


namespace gpu {

/** One past the last neighbour of node i (there are only n offsets, so the last list ends at m). */
__device__ __forceinline__
std::size_t neighbours_end( SparseGraph const & g, std::size_t i )
{
    return i + 1 < g.n ? g.neighbours_start_at[ i + 1 ] : g.m;
}

/** Inclusive prefix sum across the 32 lanes of a warp, register to register. */
__device__ __forceinline__
node_t warp_inclusive_scan( node_t x )
{
    unsigned int const lane = threadIdx.x % warpSize;
    for( unsigned int offset = 1; offset < warpSize; offset <<= 1 )
    {
        node_t const y = __shfl_up_sync( 0xffffffff, x, offset );
        if( lane >= offset ) x += y;
    }
    return x;
}

/**
 * Inclusive prefix sum across a block (blockDim.x a multiple of 32): warps scan
 * with shuffles, then warp 0 scans the per-warp totals kept in shared memory.
 * Must be called by every thread of the block. `warp_sums` is a 32-entry shared
 * array, and `total` receives the sum over the whole block.
 */
__device__
node_t block_inclusive_scan( node_t x, node_t * warp_sums, node_t & total )
{
    unsigned int const lane = threadIdx.x % warpSize;
    unsigned int const warp = threadIdx.x / warpSize;
    unsigned int const num_warps = blockDim.x / warpSize;

    x = warp_inclusive_scan( x );
    if( lane == warpSize - 1 ) warp_sums[ warp ] = x;
    __syncthreads();

    if( warp == 0 )
    {
        warp_sums[ lane ] = warp_inclusive_scan( lane < num_warps ? warp_sums[ lane ] : 0 );
    }
    __syncthreads();

    if( warp > 0 ) x += warp_sums[ warp - 1 ];
    total = warp_sums[ num_warps - 1 ];
    __syncthreads(); // warp_sums can safely be reused after this
    return x;
}

/** Threads per block of the prefix-sum kernels (one value per thread). */
constexpr unsigned int SCAN_BLOCK_SIZE = 1024;

/** Exclusive prefix sum within each block; also records each block's total. `in` and `out` may alias. */
__global__
void scan_blocks( node_t const * in, node_t * out, std::size_t n, node_t * block_sums )
{
    __shared__ node_t warp_sums[ 32 ];
    std::size_t const idx = static_cast< std::size_t >( blockIdx.x ) * blockDim.x + threadIdx.x;

    node_t const x = idx < n ? in[ idx ] : 0;
    node_t total;
    node_t const inclusive = block_inclusive_scan( x, warp_sums, total );

    if( idx < n ) out[ idx ] = inclusive - x;
    if( threadIdx.x == 0 ) block_sums[ blockIdx.x ] = total;
}

/** Adds the (already scanned) total of all preceding blocks to every value. */
__global__
void add_block_offsets( node_t * data, std::size_t n, node_t const * block_offsets )
{
    std::size_t const idx = static_cast< std::size_t >( blockIdx.x ) * blockDim.x + threadIdx.x;
    if( idx < n ) data[ idx ] += block_offsets[ blockIdx.x ];
}

/**
 * Grid-level exclusive prefix sum of n values (`d_in` and `d_out` may alias):
 * scan every block, recursively scan the block totals, then add them back.
 */
inline void exclusive_scan( node_t const * d_in, node_t * d_out, std::size_t n )
{
    unsigned int const num_blocks = static_cast< unsigned int >( ( n + SCAN_BLOCK_SIZE - 1 ) / SCAN_BLOCK_SIZE );
    node_t * d_block_sums;
    CUDA_CALL( cudaMalloc( (void**)&d_block_sums, sizeof( node_t ) * num_blocks ) );

    scan_blocks<<< num_blocks, SCAN_BLOCK_SIZE >>>( d_in, d_out, n, d_block_sums );
    CHECK_ERROR( "scan_blocks" );
    if( num_blocks > 1 )
    {
        exclusive_scan( d_block_sums, d_block_sums, num_blocks );
        add_block_offsets<<< num_blocks, SCAN_BLOCK_SIZE >>>( d_out, n, d_block_sums );
        CHECK_ERROR( "add_block_offsets" );
    }

    CUDA_CALL( cudaFree( d_block_sums ) );
}

/** Build step 1: every edge (u,v) adds one to the degree of u. */
__global__
void count_degrees( edge_t const * edge_list, std::size_t m, node_t * degree )
{
    std::size_t const stride = static_cast< std::size_t >( gridDim.x ) * blockDim.x;
    for( std::size_t e = static_cast< std::size_t >( blockIdx.x ) * blockDim.x + threadIdx.x; e < m; e += stride )
    {
        atomicAdd( &degree[ edge_list[ e ].x ], 1 );
    }
}

/**
 * Constructs a SparseGraph from an input edge list of m edges.
 * (Build step 3: every edge claims the next free slot in its source's list
 * through a per-node atomic cursor, so lists come out in arbitrary order.)
 *
 * @pre The pointers in SparseGraph g have already been allocated.
 * @pre g.neighbours_start_at holds the exclusive prefix sum of the degrees.
 */
__global__
void build_graph( SparseGraph g, edge_t const * edge_list, std::size_t m, node_t * cursor )
{
    std::size_t const stride = static_cast< std::size_t >( gridDim.x ) * blockDim.x;
    for( std::size_t e = static_cast< std::size_t >( blockIdx.x ) * blockDim.x + threadIdx.x; e < m; e += stride )
    {
        edge_t const edge = edge_list[ e ];
        node_t const slot = g.neighbours_start_at[ edge.x ] + atomicAdd( &cursor[ edge.x ], 1 );
        g.neighbours[ slot ] = edge.y;
    }
}

/**
 * Build step 4: sorts every neighbour list, one warp per node. A simple graph has
 * no duplicate neighbours, so the number of smaller neighbours is a value's slot.
 */
__global__
void sort_neighbour_lists( SparseGraph unsorted, node_t * sorted )
{
    unsigned int const lane = threadIdx.x % warpSize;
    std::size_t const warp      = ( static_cast< std::size_t >( blockIdx.x ) * blockDim.x + threadIdx.x ) / warpSize;
    std::size_t const num_warps = ( static_cast< std::size_t >( gridDim.x ) * blockDim.x ) / warpSize;

    for( std::size_t u = warp; u < unsorted.n; u += num_warps )
    {
        std::size_t const begin = unsorted.neighbours_start_at[ u ];
        std::size_t const end   = neighbours_end( unsorted, u );
        for( std::size_t p = begin + lane; p < end; p += warpSize )
        {
            node_t const v = unsorted.neighbours[ p ];
            std::size_t rank = 0;
            for( std::size_t q = begin; q < end; ++q )
            {
                rank += unsorted.neighbours[ q ] < v;
            }
            sorted[ begin + rank ] = v;
        }
    }
}

/**
  * Repopulates the adjacency lists as a new graph that represents
  * the two-hop neighbourhood of input graph g
  *
  * Not implemented: the CSR reachability query is only needed for the A+ / 100 tiers.
  */
__global__
void two_hop_reachability( SparseGraph g )
{
    // not attempted
    return;
}

/**
 * Host-side driver for build_graph(): count degrees, scan them into offsets,
 * scatter the edges, and sort each neighbour list.
 */
inline void build( SparseGraph & g, edge_t const * edge_list, std::size_t m )
{
    g.m = m;
    node_t * d_degree, * d_unsorted;
    CUDA_CALL( cudaMalloc( (void**)&d_degree,   sizeof( node_t ) * g.n ) );
    CUDA_CALL( cudaMalloc( (void**)&d_unsorted, sizeof( node_t ) * std::max< std::size_t >( m, 1 ) ) );
    unsigned int const blocks = num_blocks_for( m, DEFAULT_BLOCK_SIZE );

    // 1. degree of every node
    CUDA_CALL( cudaMemset( d_degree, 0, sizeof( node_t ) * g.n ) );
    count_degrees<<< blocks, DEFAULT_BLOCK_SIZE >>>( edge_list, m, d_degree );
    CHECK_ERROR( "count_degrees" );

    // 2. offsets = exclusive prefix sum of the degrees
    exclusive_scan( d_degree, g.neighbours_start_at, g.n );

    // 3. scatter, reusing the degree array as per-node cursors
    CUDA_CALL( cudaMemset( d_degree, 0, sizeof( node_t ) * g.n ) );
    SparseGraph const unsorted{ g.n, m, g.neighbours_start_at, d_unsorted };
    build_graph<<< blocks, DEFAULT_BLOCK_SIZE >>>( unsorted, edge_list, m, d_degree );
    CHECK_ERROR( "build_graph(SparseGraph)" );

    // 4. sort every list into its final place
    sort_neighbour_lists<<< num_blocks_for( g.n * 32, DEFAULT_BLOCK_SIZE ), DEFAULT_BLOCK_SIZE >>>( unsorted, g.neighbours );
    CHECK_ERROR( "sort_neighbour_lists" );

    CUDA_CALL( cudaFree( d_unsorted ) );
    CUDA_CALL( cudaFree( d_degree ) );
}

} // namespace gpu
} // namespace a2
} // namespace csc485b

In [ ]:
%%cuda_group_save -g "source" -n "cpu_reference.h"
/**
 * Straightforward CPU implementations used to check the GPU results.
 */
#pragma once

#include <algorithm> // std::sort, std::fill
#include <cstddef>   // std::size_t type
#include <cstdint>   // std::uint64_t
#include <vector>

#include "data_types.h"

namespace csc485b {
namespace a2      {
namespace cpu     {

/** Sorted neighbour list of every node: what both GPU graph types must contain. */
using adjacency_t = std::vector< node_list_t >;

/** Builds the sorted adjacency lists straight from the edge list. */
inline adjacency_t build_graph( edge_list_t const & edges, std::size_t n )
{
    adjacency_t adjacency( n );
    for( auto const & e : edges )
    {
        adjacency[ e.x ].push_back( e.y );
    }
    for( auto & neighbours : adjacency )
    {
        std::sort( neighbours.begin(), neighbours.end() );
    }
    return adjacency;
}

/**
 * Two-hop neighbourhood by definition: N2(i) = ( union of N(k) over k in N(i) ) \ {i}.
 * Each N(k) is stored as a bitset so that the union is a word-wise OR.
 */
inline adjacency_t two_hop_reachability( adjacency_t const & adjacency )
{
    std::size_t const n = adjacency.size();
    std::size_t const words = ( n + 63 ) / 64;

    std::vector< std::uint64_t > bitsets( n * words, 0 );
    for( std::size_t k = 0; k < n; ++k )
    {
        for( node_t const j : adjacency[ k ] )
        {
            bitsets[ k * words + j / 64 ] |= std::uint64_t{ 1 } << ( j % 64 );
        }
    }

    adjacency_t result( n );
    std::vector< std::uint64_t > reachable( words );
    for( std::size_t i = 0; i < n; ++i )
    {
        std::fill( reachable.begin(), reachable.end(), 0 );
        for( node_t const k : adjacency[ i ] )
        {
            for( std::size_t w = 0; w < words; ++w )
            {
                reachable[ w ] |= bitsets[ k * words + w ];
            }
        }
        for( std::size_t j = 0; j < n; ++j )
        {
            if( j != i && ( reachable[ j / 64 ] >> ( j % 64 ) & 1 ) )
            {
                result[ i ].push_back( static_cast< node_t >( j ) );
            }
        }
    }
    return result;
}

} // namespace cpu
} // namespace a2
} // namespace csc485b

In [ ]:
%%cuda_group_save -g "source" -n "main.cu"
/**
 * Driver for the benchmark comparison. Generates random data,
 * runs the CPU baseline, and then runs your code.
 *
 * Settings are read from environment variables (set them with %env in the notebook):
 *   N      number of nodes in the benchmark graph              (default 4096)
 *   DEG    expected_degree handed to the generator             (default 40, ~1% density at n = 4096)
 *   REPS   timed repetitions; the first one is a warm-up       (default 5)
 *   TESTS  1 = first run the worked example and the
 *          correctness sweep against the CPU reference         (default 1)
 */

#include <cstdlib>  // std::getenv
#include <iomanip>  // std::setw
#include <iostream> // std::cout, std::endl
#include <iterator> // std::ostream_iterator
#include <string>
#include <vector>

#include "dense_graph.h"
#include "sparse_graph.h"

#include "cpu_reference.h"
#include "data_generator.h"
#include "data_types.h"

namespace a2 = csc485b::a2;
using adjacency_t = a2::cpu::adjacency_t;

/** GPU time of one run(), in microseconds. */
struct Timing
{
    float build_us        = 0;
    float reachability_us = 0;
};

/** Reads a non-negative integer setting from the environment. */
std::size_t env_or( char const * name, std::size_t fallback )
{
    char const * value = std::getenv( name );
    return value ? std::stoull( value ) : fallback;
}

/** Calls the provided generator while keeping its precondition m < n(n-1). */
a2::edge_list_t generate( std::size_t n, std::size_t expected_degree )
{
    return a2::generate_graph( n, std::min( n * expected_degree, n * ( n - 1 ) - 1 ) );
}

/** Copies an edge list to the device. */
a2::edge_t * upload( a2::edge_list_t const & edges )
{
    a2::edge_t * d_edges;
    CUDA_CALL( cudaMalloc( (void**)&d_edges, sizeof( a2::edge_t ) * std::max< std::size_t >( edges.size(), 1 ) ) );
    CUDA_CALL( cudaMemcpy( d_edges, edges.data(), sizeof( a2::edge_t ) * edges.size(), cudaMemcpyHostToDevice ) );
    return d_edges;
}

a2::DenseGraph allocate_dense( std::size_t n )
{
    a2::node_t * d_matrix;
    CUDA_CALL( cudaMalloc( (void**)&d_matrix, sizeof( a2::node_t ) * n * n ) );
    return a2::DenseGraph{ n, d_matrix };
}

a2::SparseGraph allocate_sparse( std::size_t n, std::size_t m )
{
    a2::node_t * d_offsets, * d_neighbours;
    CUDA_CALL( cudaMalloc( (void**)&d_offsets,    sizeof( a2::node_t ) * n ) );
    CUDA_CALL( cudaMalloc( (void**)&d_neighbours, sizeof( a2::node_t ) * std::max< std::size_t >( m, 1 ) ) );
    return a2::SparseGraph{ n, m, d_offsets, d_neighbours };
}

void free_graph( a2::DenseGraph const & g )
{
    CUDA_CALL( cudaFree( g.adjacencyMatrix ) );
}

void free_graph( a2::SparseGraph const & g )
{
    CUDA_CALL( cudaFree( g.neighbours ) );
    CUDA_CALL( cudaFree( g.neighbours_start_at ) );
}

/** Copies the adjacency matrix back as lists; a cell that is not 0 or 1 shows up as -1 and fails any comparison. */
adjacency_t download( a2::DenseGraph const & g )
{
    std::vector< a2::node_t > matrix( g.matrix_size() );
    CUDA_CALL( cudaMemcpy( matrix.data(), g.adjacencyMatrix, sizeof( a2::node_t ) * matrix.size(), cudaMemcpyDeviceToHost ) );

    adjacency_t lists( g.n );
    for( std::size_t i = 0; i < g.n; ++i )
    {
        for( std::size_t j = 0; j < g.n; ++j )
        {
            a2::node_t const cell = matrix[ i * g.n + j ];
            if( cell != 0 ) lists[ i ].push_back( cell == 1 ? static_cast< a2::node_t >( j ) : -1 );
        }
    }
    return lists;
}

/** Copies the CSR arrays back as lists; malformed offsets give an empty result, which fails any comparison. */
adjacency_t download( a2::SparseGraph const & g )
{
    std::vector< a2::node_t > offsets( g.n ), neighbours( g.m );
    CUDA_CALL( cudaMemcpy( offsets.data(),    g.neighbours_start_at, sizeof( a2::node_t ) * g.n, cudaMemcpyDeviceToHost ) );
    CUDA_CALL( cudaMemcpy( neighbours.data(), g.neighbours,          sizeof( a2::node_t ) * g.m, cudaMemcpyDeviceToHost ) );

    adjacency_t lists( g.n );
    for( std::size_t i = 0; i < g.n; ++i )
    {
        long long const begin = offsets[ i ];
        long long const end   = i + 1 < g.n ? offsets[ i + 1 ] : static_cast< long long >( g.m );
        if( begin < 0 || begin > end || end > static_cast< long long >( g.m ) ) return {};
        lists[ i ].assign( neighbours.begin() + begin, neighbours.begin() + end );
    }
    return lists;
}

void print( a2::DenseGraph const & g )
{
    std::vector< a2::node_t > matrix( g.matrix_size() );
    CUDA_CALL( cudaMemcpy( matrix.data(), g.adjacencyMatrix, sizeof( a2::node_t ) * matrix.size(), cudaMemcpyDeviceToHost ) );
    for( std::size_t i = 0; i < g.n; ++i )
    {
        std::cout << "    [ ";
        std::copy( matrix.cbegin() + i * g.n, matrix.cbegin() + ( i + 1 ) * g.n, std::ostream_iterator< a2::node_t >( std::cout, " " ) );
        std::cout << "]\n";
    }
}

void print( a2::SparseGraph const & g )
{
    std::vector< a2::node_t > offsets( g.n ), neighbours( g.m );
    CUDA_CALL( cudaMemcpy( offsets.data(),    g.neighbours_start_at, sizeof( a2::node_t ) * g.n, cudaMemcpyDeviceToHost ) );
    CUDA_CALL( cudaMemcpy( neighbours.data(), g.neighbours,          sizeof( a2::node_t ) * g.m, cudaMemcpyDeviceToHost ) );
    std::cout << "    neighbours_start_at: [ ";
    std::copy( offsets.cbegin(), offsets.cend(), std::ostream_iterator< a2::node_t >( std::cout, " " ) );
    std::cout << "]\n    neighbours:          [ ";
    std::copy( neighbours.cbegin(), neighbours.cend(), std::ostream_iterator< a2::node_t >( std::cout, " " ) );
    std::cout << "]\n";
}

/**
 * Runs timing tests on a DenseGraph implementation.
 * Consists of independently constructing the graph and then
 * modifying it to its two-hop neighbourhood.
 */
template < typename DeviceGraph, typename... Options >
Timing run( DeviceGraph & g, a2::edge_t const * d_edges, std::size_t m, Options... options )
{
    cudaEvent_t build_start, reachability_start, end;
    CUDA_CALL( cudaEventCreate( &build_start ) );
    CUDA_CALL( cudaEventCreate( &reachability_start ) );
    CUDA_CALL( cudaEventCreate( &end ) );

    cudaDeviceSynchronize();
    cudaEventRecord( build_start );

    a2::gpu::build( g, d_edges, m );

    cudaEventRecord( reachability_start );

    a2::gpu::two_hop( g, options... );

    cudaEventRecord( end );
    cudaEventSynchronize( end );
    CHECK_ERROR( "run" );

    Timing timing;
    float ms;
    cudaEventElapsedTime( &ms, build_start, reachability_start );
    timing.build_us = 1000 * ms;
    cudaEventElapsedTime( &ms, reachability_start, end );
    timing.reachability_us = 1000 * ms;

    cudaEventDestroy( build_start );
    cudaEventDestroy( reachability_start );
    cudaEventDestroy( end );
    return timing;
}

/**
 * Allocates space for a dense graph and then runs the test code on it.
 * If `result` is given, the final graph is copied back into it.
 */
Timing run_dense( a2::edge_t const * d_edges, std::size_t n, std::size_t m, a2::gpu::DenseAlgorithm algorithm, adjacency_t * result = nullptr )
{
    a2::DenseGraph d_dg = allocate_dense( n );
    Timing const timing = run( d_dg, d_edges, m, algorithm );
    if( result ) *result = download( d_dg );
    free_graph( d_dg );
    return timing;
}

/** Builds a graph on the GPU and returns what ended up on the device. */
template < typename DeviceGraph >
adjacency_t gpu_build_result( DeviceGraph g, a2::edge_t const * d_edges, std::size_t m )
{
    a2::gpu::build( g, d_edges, m );
    adjacency_t const lists = download( g );
    free_graph( g );
    return lists;
}

/** Builds a DenseGraph on the GPU, runs the two-hop query, and returns the result. */
adjacency_t gpu_two_hop_result( a2::DenseGraph g, a2::edge_t const * d_edges, std::size_t m, a2::gpu::DenseAlgorithm algorithm )
{
    a2::gpu::build( g, d_edges, m );
    a2::gpu::two_hop( g, algorithm );
    adjacency_t const lists = download( g );
    free_graph( g );
    return lists;
}

/** Reproduces the example from the assignment description. */
void worked_example()
{
    std::cout << "=== Worked example from the assignment: edges (0,1), (0,2), (2,3) ===\n";
    a2::edge_list_t const edges{ make_int2( 0, 1 ), make_int2( 1, 0 ), make_int2( 0, 2 ),
                                 make_int2( 2, 0 ), make_int2( 2, 3 ), make_int2( 3, 2 ) };
    std::size_t const n = 4;
    std::size_t const m = edges.size();
    a2::edge_t * d_edges = upload( edges );

    a2::DenseGraph dense = allocate_dense( n );
    a2::gpu::build( dense, d_edges, m );
    std::cout << "DenseGraph after build_graph():\n";
    print( dense );
    a2::gpu::two_hop( dense );
    std::cout << "DenseGraph after two_hop_reachability():\n";
    print( dense );
    free_graph( dense );

    a2::SparseGraph sparse = allocate_sparse( n, m );
    a2::gpu::build( sparse, d_edges, m );
    std::cout << "SparseGraph after build_graph():\n";
    print( sparse );
    free_graph( sparse );

    CUDA_CALL( cudaFree( d_edges ) );
    std::cout << std::endl;
}

/**
 * Checks every GPU function against the CPU reference on a range of sizes and
 * densities, including sizes that are not multiples of the block/tile sizes.
 */
bool correctness_sweep()
{
    struct Case { std::size_t n, expected_degree; };
    std::vector< Case > const cases{ { 2, 1 }, { 8, 4 }, { 33, 1 }, { 33, 16 }, { 100, 1 }, { 100, 50 },
                                     { 257, 3 }, { 257, 128 }, { 1000, 10 }, { 1000, 500 }, { 1500, 30 } };

    std::cout << "=== Correctness sweep: GPU vs CPU reference ===\n";
    bool all_correct = true;
    for( auto const & c : cases )
    {
        a2::edge_list_t const edges = generate( c.n, c.expected_degree );
        std::size_t const m = edges.size();
        a2::edge_t * d_edges = upload( edges );

        adjacency_t const graph   = a2::cpu::build_graph( edges, c.n );
        adjacency_t const two_hop = a2::cpu::two_hop_reachability( graph );

        auto check = [ & ]( char const * name, adjacency_t const & actual, adjacency_t const & expected )
        {
            bool const correct = actual == expected;
            all_correct = all_correct && correct;
            std::cout << "  " << name << ( correct ? " ok " : " FAIL" );
        };

        std::cout << "n=" << std::setw( 5 ) << c.n << " m=" << std::setw( 7 ) << m << " |";
        check( "dense build",  gpu_build_result( allocate_dense( c.n ), d_edges, m ), graph );
        check( "sparse build", gpu_build_result( allocate_sparse( c.n, m ), d_edges, m ), graph );
        check( "| two-hop: dense naive", gpu_two_hop_result( allocate_dense( c.n ), d_edges, m, a2::gpu::DenseAlgorithm::naive ), two_hop );
        check( "dense tiled",  gpu_two_hop_result( allocate_dense( c.n ), d_edges, m, a2::gpu::DenseAlgorithm::tiled ), two_hop );
        check( "dense warp",   gpu_two_hop_result( allocate_dense( c.n ), d_edges, m, a2::gpu::DenseAlgorithm::warp ), two_hop );
        std::cout << std::endl;

        CUDA_CALL( cudaFree( d_edges ) );
    }
    std::cout << ( all_correct ? "All checks passed." : "SOME CHECKS FAILED!" ) << "\n" << std::endl;
    return all_correct;
}

/** Times every implementation on one generated graph and checks each final result against the CPU. */
bool benchmark( std::size_t n, std::size_t expected_degree, std::size_t reps )
{
    // Create input
    a2::edge_list_t const graph = generate( n, expected_degree );
    std::size_t const m = graph.size();

    // lazily echo out input graph (only when it is small enough to read)
    if( n <= 16 )
    {
        for( auto const& e : graph )
        {
            std::cout << "(" << e.x << "," << e.y << ") ";
        }
        std::cout << std::endl;
    }

    adjacency_t const expected = a2::cpu::two_hop_reachability( a2::cpu::build_graph( graph, n ) );
    std::size_t two_hop_m = 0;
    for( auto const & neighbours : expected ) two_hop_m += neighbours.size();

    double const cells = static_cast< double >( n ) * ( n - 1 );
    std::cout << "=== Benchmark: n=" << n << ", m=" << m << " (density " << std::setprecision( 3 ) << 100 * m / cells
              << "%) -> two-hop m=" << two_hop_m << " (density " << 100 * two_hop_m / cells << "%), "
              << reps << " reps, first one is a warm-up ===\n";

    // allocate and memcpy input to device
    a2::edge_t * d_edges = upload( graph );

    struct Result { std::string name; Timing time; bool correct; };
    std::vector< Result > results;
    auto measure = [ & ]( std::string const & name, auto && run_once )
    {
        Timing total;
        std::size_t counted = 0;
        adjacency_t output;
        for( std::size_t r = 0; r < reps; ++r )
        {
            Timing const t = run_once( r + 1 == reps ? &output : nullptr );
            if( reps == 1 || r > 0 )
            {
                total.build_us        += t.build_us;
                total.reachability_us += t.reachability_us;
                ++counted;
            }
        }
        total.build_us        /= counted;
        total.reachability_us /= counted;
        results.push_back( { name, total, output == expected } );
    };

    // run your code!
    using a2::gpu::DenseAlgorithm;
    measure( "Dense naive (C/B-)", [ & ]( adjacency_t * out ) { return run_dense( d_edges, n, m, DenseAlgorithm::naive, out ); } );
    measure( "Dense tiled (B)",    [ & ]( adjacency_t * out ) { return run_dense( d_edges, n, m, DenseAlgorithm::tiled, out ); } );
    measure( "Dense warp (A-)",    [ & ]( adjacency_t * out ) { return run_dense( d_edges, n, m, DenseAlgorithm::warp,  out ); } );

    std::cout << std::fixed << std::setprecision( 1 )
              << std::left  << std::setw( 20 ) << "version"
              << std::right << std::setw( 14 ) << "build (us)" << std::setw( 16 ) << "two-hop (us)"
              << std::setw( 18 ) << "vs dense naive" << std::setw( 10 ) << "correct" << "\n";
    bool all_correct = true;
    for( auto const & r : results )
    {
        all_correct = all_correct && r.correct;
        std::cout << std::left  << std::setw( 20 ) << r.name
                  << std::right << std::setw( 14 ) << r.time.build_us << std::setw( 16 ) << r.time.reachability_us
                  << std::setw( 17 ) << results[ 0 ].time.reachability_us / r.time.reachability_us << "x"
                  << std::setw( 10 ) << ( r.correct ? "yes" : "NO" ) << "\n";
    }
    std::cout << std::defaultfloat << std::endl;

    CUDA_CALL( cudaFree( d_edges ) );
    return all_correct;
}

int main()
{
    std::size_t const n               = env_or( "N", 4096 );
    std::size_t const expected_degree = env_or( "DEG", 40 );
    std::size_t const reps            = std::max< std::size_t >( env_or( "REPS", 5 ), 1 );

    bool correct = true;
    if( env_or( "TESTS", 1 ) )
    {
        worked_example();
        correct = correctness_sweep();
    }
    correct = benchmark( n, expected_degree, reps ) && correct;

    return correct ? EXIT_SUCCESS : EXIT_FAILURE;
}

## Correctness check and benchmark (n = 4096)

Settings are passed with `%env`: `N` (nodes), `DEG` (expected degree for the generator), `REPS` (timed runs, the first one is a warm-up) and `TESTS` (1 = run the example and the correctness check first). Times are measured with `cudaEvent`s and include the allocations inside `build()` / `two_hop()`.

In [ ]:
%env N=4096
%env DEG=40
%env REPS=5
%env TESTS=1

In [ ]:
%cuda_group_run --group "source" --compiler-args "-O3 -g -std=c++20 -arch=sm_75"

## Profiling with Nsight Compute

Profiles only the two-hop kernels (one run, no tests). The timing table printed by this cell is not meaningful, because ncu replays each kernel and locks the clock; use the benchmark above for times. The interesting values are SM / memory throughput, achieved occupancy and `smsp__thread_inst_executed_per_inst_executed.ratio` (active threads per warp instruction).

If Colab says `ERR_NVGPUCTRPERM`, the counters are not available on that runtime (Kaggle's T4 works).

In [ ]:
%env N=4096
%env DEG=40
%env REPS=1
%env TESTS=0
%cuda_group_run --group "source" --compiler-args "-O3 -lineinfo -std=c++20 -arch=sm_75" --profile --profiler ncu --profiler-args "--kernel-name regex:two_hop|pack_rows --section SpeedOfLight --section Occupancy --section LaunchStats --section MemoryWorkloadAnalysis --metrics smsp__thread_inst_executed_per_inst_executed.ratio"

## Why the warp-level version is faster (A tier)

Profiled with Nsight Compute on the T4 (n = 4096, ~1 % density). ncu locks the clock at 585 MHz, so durations are longer than in the benchmark above.

| Kernel | Duration | SM throughput | L1/TEX throughput | L2 hit rate | DRAM throughput | Occupancy | Threads per warp instr. |
|---|---|---|---|---|---|---|---|
| Naive | 584 ms | 63 % | 94 % | 53 % | 19 % | ~100 % | 32 |
| Tiled | 372 ms | 74 % | 95 % | 50 % | 28 % | ~100 % | 32 |
| pack_rows + warp | 1.0 + 8.2 ms | 95 % | 95 % | 99 % | 4 % | 99 % | 32 |

Benchmark (normal clock): naive 296 ms, tiled 175 ms (1.7×), warp-level 3.65 ms (81×).

**Naive (C / B-).** One thread per output cell, so n² threads. The profiler shows 32 active threads per warp instruction, so nothing is predicated off; that is why we use the same kernel for C and B-. The problem is that every multiply-add needs two load instructions. Most of them hit in L1 (86 %), so DRAM is not the limit (19 %), but the L1/TEX pipeline that serves the loads is full (94 %).

**Tiled (B).** 16×16 tiles in shared memory with `__syncthreads()`, so each value from global memory is reused 16 times. It only gives 1.7× because every multiply-add still reads two values from shared memory, and on Turing shared memory goes through the same L1/TEX pipeline (still 95 %).

**Warp-level (A-).** To get faster we need fewer and wider operations, not just a different memory. The matrix only has 0s and 1s and we only need to know if the sum is > 0, so we store 32 cells in one 32-bit word and replace multiply-add with AND/OR. One AND checks 32 values of k at once, so there are 32× fewer inner operations (n³/32 ≈ 2.1 G instead of 68.7 G).
- `__ballot_sync` packs the matrix: each lane checks one cell and the warp gets one 32-bit word in one instruction. The packed matrix is 2 MB and fits in the 4 MB L2 (99 % hit rate, DRAM only 4 % busy).
- The graph is undirected, so column j = row j. A²[i][j] > 0 means rows i and j share a set bit, so both operands are packed rows.
- Each warp computes 4 output rows and keeps its row words in registers. `__shfl_sync` broadcasts them to the other lanes, so the row side needs no memory loads at all.
- The column words are in shared memory, padded to `[32][33]`, so lane l reading `col_words[l][t]` hits bank (l + t) mod 32: no bank conflicts. Each shared-memory word is used for 4 rows.
- Lane l writes output column j0 + l, so the final writes are coalesced.

The kernel goes from load-bound (naive, tiled) to compute-bound (95 % SM throughput) at 99 % occupancy. Most of the speed-up comes from bit-packing, which `__ballot_sync` makes cheap; `__shfl_sync` keeps the extra row loads out of the L1/shared-memory pipeline. We did not time a version without shuffles.
